# FE101 B1 1965 — 5 radial × 5 axial CLEAN depletion preparation

Separate notebook based on verified initial radial-case sources. Physical B,
25 independent fuel materials, original composition/density/temperature/S(a,b).
All other cells and materials preserved. No OpenMC transport/depletion during
preparation. Preliminary CLEAN: 5000 particles, 30 batches, 10 inactive.

Radial edges use 1.791*sqrt(i/5) without rounding. Axial edges are 10.200,
17.312, 24.424, 31.536, 38.648, 45.760 cm. Each material occurs once.


In [ ]:
from pathlib import Path
RUN_DEPLETION = False
RUN_POSTPROCESS = False
RUN_BENCHMARKS = False
EXPORT_PREPARED_INPUTS = True
CASE_DIR = Path(globals().get('FE101_CASE_DIR', Path.cwd()))
if not (CASE_DIR/'prepare_geometry.py').is_file():
    CASE_DIR = next(parent/'z3st/cases/verification/neutronics/fe101_5x5_depletion'
                    for parent in [Path.cwd(), *Path.cwd().parents]
                    if (parent/'z3st/cases/verification/neutronics/fe101_5x5_depletion/prepare_geometry.py').is_file())
RADIAL_CASE_DIR = CASE_DIR.parent/'fe101_radial_depletion'
RADIAL_NOTEBOOK = RADIAL_CASE_DIR/'triga_single_FE101_B1_1965_5RINGS_CLEAN.ipynb'
OUTPUT_ROOT = Path(globals().get('FE101_OUTPUT_ROOT', CASE_DIR/'preparation'))
PREPARED_DIR = OUTPUT_ROOT/'prepared_inputs'
RUN_DIR = OUTPUT_ROOT/'run_3500h_ED'


In [ ]:
"""Deterministic object construction and checks only; no OpenMC execution."""
import hashlib
import json
import os
import runpy
from pathlib import Path
import numpy as np
import openmc

initial = runpy.run_path(str(RADIAL_CASE_DIR/'prepare_initial.py'))['prepare_initial'](export=False)
baseline_audit = initial['audit']
protected_paths = [RADIAL_NOTEBOOK, RADIAL_CASE_DIR/'radial_geometry.py', RADIAL_CASE_DIR.parent/'fe101_geometry_static/source_model.py']
protected_hashes = {str(p.relative_to(RADIAL_CASE_DIR.parent)):hashlib.sha256(p.read_bytes()).hexdigest() for p in protected_paths}
radial_model = initial['radial_model']
geometry, materials_file = radial_model.geometry, radial_model.materials
settings_file = radial_model.settings
tallies_file = radial_model.tallies
all_cells_before = geometry.get_all_cells()
old_cell_ids = {r['cell_id'] for r in baseline_audit['rings']}
old_material_ids = {r['material_id'] for r in baseline_audit['rings']}
unchanged_cells = {cid:(str(c.region),c.fill_type,getattr(c.fill,'id',None),c.temperature)
                   for cid,c in all_cells_before.items() if cid not in old_cell_ids}
unchanged_materials = {m.id:(m.nuclides.copy(),m.density,m.density_units,m.temperature,list(m._sab),m.depletable)
                       for m in materials_file if m.id not in old_material_ids}
target = next(u for u in geometry.get_all_universes().values() if old_cell_ids <= set(u.cells))
R_FUEL_CM=1.791
radial_edges_cm=R_FUEL_CM*np.sqrt(np.arange(6)/5.0)
axial_edges_cm=np.array([10.200,17.312,24.424,31.536,38.648,45.760])
assert np.allclose(np.diff(axial_edges_cm),7.112,rtol=0,atol=1e-13)
assert np.allclose(radial_edges_cm,[baseline_audit['rings'][0]['r_in_cm']]+[r['r_out_cm'] for r in baseline_audit['rings']],rtol=0,atol=0)
# Reuse exact existing end planes; only add internal axial partition planes.
first_cell=all_cells_before[baseline_audit['rings'][0]['cell_id']]
planes={s.z0:s for s in first_cell.region.get_surfaces().values() if isinstance(s,openmc.ZPlane)}
axial_planes=[planes[axial_edges_cm[0]]]+[openmc.ZPlane(z0=float(z)) for z in axial_edges_cm[1:-1]]+[planes[axial_edges_cm[-1]]]
domain_materials=[]
domain_cells=[]
domains=[]
for ir,ring in enumerate(baseline_audit['rings'],1):
    old_cell=all_cells_before[ring['cell_id']]
    source_material=old_cell.fill
    surfaces=list(old_cell.region.get_surfaces().values())
    cylinders=[s for s in surfaces if isinstance(s,openmc.ZCylinder)]
    outer=next(s for s in cylinders if s.r==radial_edges_cm[ir])
    radial_region=-outer
    if ir>1:
        inner=next(s for s in cylinders if s.r==radial_edges_cm[ir-1])
        radial_region=radial_region & +inner
    target.remove_cell(old_cell)
    materials_file.remove(source_material)
    for iz in range(1,6):
        mat=source_material.clone()
        mat.name=f'Fuel101_B1_r{ir}_z{iz}'
        mat.depletable=True
        volume=float(np.pi*(radial_edges_cm[ir]**2-radial_edges_cm[ir-1]**2)*(axial_edges_cm[iz]-axial_edges_cm[iz-1]))
        mat.volume=volume
        cell=openmc.Cell(name=f'FE101 B1 active r{ir} z{iz}',fill=mat,
                         region=radial_region & +axial_planes[iz-1] & -axial_planes[iz])
        target.add_cell(cell)
        materials_file.append(mat)
        domain_materials.append(mat)
        domain_cells.append(cell)
        assert mat.nuclides==source_material.nuclides
        assert mat.density==6.3 and mat.density_units=='g/cm3'
        assert mat.temperature==source_material.temperature and mat._sab==source_material._sab
        densities=mat.get_nuclide_atom_densities()
        initial_atoms={n:float(d*1e24*volume) for n,d in densities.items()}
        element=lambda n: ''.join(c for c in n if c.isalpha())
        metal_atoms=sum(a for n,a in initial_atoms.items() if element(n) in ('U','Zr'))
        u_mass=sum(mat.get_mass(n) for n in densities if element(n)=='U')/1000
        hzr=sum(d for n,d in densities.items() if element(n)=='H')/sum(d for n,d in densities.items() if element(n)=='Zr')
        domains.append({'radial_index':ir,'axial_index':iz,'material_id':mat.id,'material_name':mat.name,'cell_id':cell.id,
                        'r_in_cm':float(radial_edges_cm[ir-1]),'r_out_cm':float(radial_edges_cm[ir]),
                        'r_in_over_R':float(radial_edges_cm[ir-1]/R_FUEL_CM),'r_out_over_R':float(radial_edges_cm[ir]/R_FUEL_CM),
                        'z_low_cm':float(axial_edges_cm[iz-1]),'z_high_cm':float(axial_edges_cm[iz]),'volume_cm3':volume,
                        'mass_g':mat.get_mass(),'initial_U_mass_kg':u_mass,'initial_U_Zr_atoms':metal_atoms,
                        'initial_atoms_by_nuclide':initial_atoms,'H_Zr_atom_ratio':hzr,'temperature_K':mat.temperature})
geometry.determine_paths(instances_only=True)
assert len({m.id for m in domain_materials})==25
assert len([m for m in materials_file if m.depletable])==25
assert {m.id for m in geometry.get_all_materials().values() if m.depletable}=={m.id for m in domain_materials}
assert all(m.num_instances==c.num_instances==1 for m,c in zip(domain_materials,domain_cells))
for d,m in zip(domains,domain_materials): d['instances']=m.num_instances
for cid,snapshot in unchanged_cells.items():
    c=geometry.get_all_cells()[cid]
    assert (str(c.region),c.fill_type,getattr(c.fill,'id',None),c.temperature)==snapshot
for mid,snapshot in unchanged_materials.items():
    m=next(m for m in materials_file if m.id==mid)
    assert (m.nuclides,m.density,m.density_units,m.temperature,list(m._sab),m.depletable)==snapshot
    assert not m.depletable
original_volume=baseline_audit['original_volume_cm3']
assert np.allclose([d['volume_cm3'] for d in domains],original_volume/25,rtol=1e-14)
assert np.isclose(sum(d['volume_cm3'] for d in domains),original_volume,rtol=1e-14)
assert np.isclose(original_volume,358.346194419,rtol=0,atol=5e-10)
original_mass=baseline_audit['original_mass_g']
original_u_mass=sum(r['initial_U_mass_kg'] for r in baseline_audit['rings'])
assert np.isclose(sum(d['mass_g'] for d in domains),original_mass,rtol=1e-14)
assert np.isclose(sum(d['initial_U_mass_kg'] for d in domains),original_u_mass,rtol=1e-14)
nuclide_checks={}
for nuc in baseline_audit['rings'][0]['initial_atoms_by_nuclide']:
    expected=sum(r['initial_atoms_by_nuclide'][nuc] for r in baseline_audit['rings'])
    actual=sum(d['initial_atoms_by_nuclide'][nuc] for d in domains)
    assert np.isclose(actual,expected,rtol=1e-14)
    nuclide_checks[nuc]={'original_atoms':expected,'reconstructed_atoms':actual,'relative_error':(actual-expected)/expected}
radial_volumes=[sum(d['volume_cm3'] for d in domains if d['radial_index']==ir) for ir in range(1,6)]
axial_volumes=[sum(d['volume_cm3'] for d in domains if d['axial_index']==iz) for iz in range(1,6)]
assert np.allclose(radial_volumes,[r['volume_cm3'] for r in baseline_audit['rings']],rtol=1e-14)
assert np.allclose(axial_volumes,original_volume/5,rtol=1e-14)
origin=np.asarray(baseline_audit['B1_origin_cm'])
for d,mat in zip(domains,domain_materials):
    rad=np.sqrt((d['r_in_cm']**2+d['r_out_cm']**2)/2)
    for phi in (0.,0.7,2.,4.):
        for zz in (d['z_low_cm']+1e-6,(d['z_low_cm']+d['z_high_cm'])/2,d['z_high_cm']-1e-6):
            assert geometry.find(tuple(origin+[rad*np.cos(phi),rad*np.sin(phi),zz]))[-1].fill is mat
for zz in (10.200001,17.312,27.98,38.648,45.759999):
    for rad,expected_name in [(1.791001,'He4'),(1.803999,'He4'),(1.804001,'Al'),(1.879999,'Al')]:
        mat=geometry.find(tuple(origin+[rad,0,zz]))[-1].fill
        if expected_name=='He4': assert mat.nuclides[0].name=='He4' and mat.density==1e-10
        else: assert mat.id==geometry.find(tuple(origin+[1.85,0,27.98]))[-1].fill.id
ordinary_fuel=next(m for m in materials_file if m.name=='Fuel 101')
assert ordinary_fuel.num_instances==60 and not ordinary_fuel.depletable
# Replace only radial material diagnostics; other original diagnostics remain.
tallies_file=openmc.Tallies([t for t in tallies_file if not t.name.startswith('FE101 B1 ring')])
domain_filter=openmc.MaterialFilter(domain_materials, filter_id=max(f.id for t in radial_model.tallies for f in t.filters)+1)
for name,score in [('FE101 B1 domain total fission','fission'),('FE101 B1 domain energy deposition','heating-local')]:
    t=openmc.Tally(name=name)
    t.filters=[domain_filter]
    t.scores=[score]
    if score=='fission': t.estimator='tracklength'
    tallies_file.append(t)
radial_model.tallies=tallies_file
assert (settings_file.particles,settings_file.batches,settings_file.inactive)==(5000,30,10)
openmc.config['cross_sections']=baseline_audit['cross_sections']
openmc.config['chain_file']=baseline_audit['chain_file']
audit={'geometry_baseline':'physical B','fuel_outer_cm':1.791,'clad_inner_cm':1.804,'clad_outer_cm':1.880,
       'gap_radial_um':130.,'radial_edges_cm':radial_edges_cm.tolist(),'axial_edges_cm':axial_edges_cm.tolist(),
       'B1_origin_cm':origin.tolist(),'domains':domains,'depletable_materials':25,'ordinary_FE101_instances':60,
       'original_volume_cm3':original_volume,'reconstructed_volume_cm3':sum(d['volume_cm3'] for d in domains),
       'original_mass_g':original_mass,'reconstructed_mass_g':sum(d['mass_g'] for d in domains),
       'original_U_mass_kg':original_u_mass,'reconstructed_U_mass_kg':sum(d['initial_U_mass_kg'] for d in domains),
       'nuclide_conservation':nuclide_checks,'radial_group_volumes_cm3':radial_volumes,'axial_group_volumes_cm3':axial_volumes,
       'cross_sections':baseline_audit['cross_sections'],'chain_file':baseline_audit['chain_file'],
       'settings':baseline_audit['settings'],'normalization_mode':'energy-deposition','diff_burnable_mats':False,
       'write_rates':True,'final_step':True,'power_W':250000.,'steps':20,'step_hours':175.,
       'protected_sha256':protected_hashes,'checks_passed':True,'transport_executed':False,'depletion_executed':False,
       'radial_benchmark':os.path.relpath(RADIAL_CASE_DIR/'reference/radial_history.json',PREPARED_DIR),
       'static_benchmark':os.path.relpath(RADIAL_CASE_DIR.parent/'fe101_axial_static/reference/static_5x5.json',PREPARED_DIR)}
assert (PREPARED_DIR/audit['radial_benchmark']).resolve().is_file() and (PREPARED_DIR/audit['static_benchmark']).resolve().is_file()
if EXPORT_PREPARED_INPUTS:
    PREPARED_DIR.mkdir(parents=True,exist_ok=True)
    geometry.export_to_xml(PREPARED_DIR/'geometry.xml')
    materials_file.export_to_xml(PREPARED_DIR/'materials.xml')
    settings_file.export_to_xml(PREPARED_DIR/'settings.xml')
    tallies_file.export_to_xml(PREPARED_DIR/'tallies.xml')
    (PREPARED_DIR/'preparation_audit.json').write_text(json.dumps(audit,indent=2)+'\n')
    import pandas as pd
    pd.DataFrame(domains).drop(columns='initial_atoms_by_nuclide').to_csv(PREPARED_DIR/'domains.csv',index=False)
for p,h in protected_hashes.items(): assert hashlib.sha256((RADIAL_CASE_DIR.parent/p).read_bytes()).hexdigest()==h
print('PASS: 25 unique materials/cells, one instance each; all conservation and geometry checks passed')
print(f'Volume: {original_volume:.12f} cm3; domain: {original_volume/25:.12f} cm3')
print(f'Fuel mass: {original_mass:.12f} g; U mass: {original_u_mass:.12f} kg')


## Depletion — disabled

Predictor, 20 × 175 h at 250000 W whole-reactor power; energy-deposition,
diff_burnable_mats=False, write_rates=True, final_step=True. This creates
21 time points, including genuine final rates. Do not differentiate materials:
the 25 physical domains already have distinct material IDs. Require a fresh
run directory. No operator is constructed unless RUN_DEPLETION is enabled.


In [ ]:
if RUN_DEPLETION:
    import openmc.deplete
    import openmc.deplete.pool
    openmc.deplete.pool.USE_MULTIPROCESSING = False
    if RUN_DIR.exists() and any(RUN_DIR.iterdir()):
        raise FileExistsError(f'Use a fresh depletion directory: {RUN_DIR}')
    if not (PREPARED_DIR/'preparation_audit.json').is_file():
        raise FileNotFoundError('Export preparation audit before running')
    RUN_DIR.mkdir(parents=True,exist_ok=True)
    operator = openmc.deplete.CoupledOperator(radial_model,chain_file=audit['chain_file'],
                 normalization_mode='energy-deposition',diff_burnable_mats=False)
    operator.output_dir=RUN_DIR
    assert len(operator.burnable_mats)==25
    assert set(operator.burnable_mats)=={str(m.id) for m in domain_materials}
    integrator=openmc.deplete.PredictorIntegrator(operator,timesteps=[175.]*20,
                     power=250000.,timestep_units='h')
    integrator.integrate(path='depletion_results_FE101_B1_1965_5x5_3500h_ED.h5',
                         write_rates=True,final_step=True)
else:
    print('Transport/depletion disabled: preparation only.')


## Future postprocessing and benchmarks — disabled

525 domain/time records. FIMA = cumulative fissions / fixed initial U+Zr
atoms, excluding H; %FIMA = 100*FIMA. Sum all chain nuclides with fission,
including U238 and actinides formed later. Rates are physical fissions/s.
BU = heating-local deposited energy / (8.64e10 * initial U mass in kg),
kept separate from FIMA. Both integrations use first-order BOS left endpoints,
consistent with Predictor; final transport supplies rates, not another interval.
Statistical and time-integration errors are not propagated.

Future zest_FIMA_2D.csv has times, radial/axial indices, radii, z bounds and
FIMA fraction. It is not fabricated during preparation.

Benchmarks aggregate along z to compare with the existing radial depletion,
compare initial fission/power patterns with the static 5×5 tally, and check
weighted FE FIMA/BU reconstruction. FIMA(0)=0, so the initial FIMA pattern
uses dFIMA/dt=fission_rate/N_initial, not normalization of zero inventories.


In [ ]:
"""Prepared postprocessing; runs only on explicit invocation after depletion."""
def postprocess_2d(run_directory, preparation_audit):
    import json
    from pathlib import Path
    import numpy as np
    import pandas as pd
    import openmc
    import openmc.deplete

    run_directory = Path(run_directory)
    metadata = json.loads(Path(preparation_audit).read_text())
    results_path = run_directory / "depletion_results_FE101_B1_1965_5x5_3500h_ED.h5"
    results = openmc.deplete.Results(results_path)
    times = np.asarray(results.get_times(time_units="s"), dtype=float)
    assert len(times) == 21 and np.all(np.diff(times) > 0)
    assert np.isclose(times[-1], 3500*3600)
    ring_metadata = metadata["domains"]
    assert set(results[0].index_mat) == {str(r["material_id"]) for r in ring_metadata}
    # Read the same chain used by the run; include fast-fission contributors,
    # e.g. U238, and actinides formed during irradiation, not only U235/Pu239.
    chain = openmc.deplete.Chain.from_xml(metadata["chain_file"])
    fission_nuclides = [n.name for n in chain.nuclides
                       if any(reaction.type == "fission" for reaction in n.reactions)]
    with openmc.StatePoint(run_directory / "openmc_simulation_n0.h5") as sp:
        score = sp.get_tally(name="FE101 B1 domain energy deposition").scores[0]
        assert score == "heating-local"
    power_by_ring = np.zeros((len(times), 25))
    tally_fission_by_ring = np.zeros_like(power_by_ring)
    heat_by_domain = np.zeros_like(power_by_ring)
    # Predictor saves BOS statepoints n0..n19 and the final transport n20.
    for k in range(len(times)):
        path = run_directory / f"openmc_simulation_n{k}.h5"
        with openmc.StatePoint(path) as sp:
            global_heat = float(sp.get_tally(name="factor-for-normalization").mean.ravel()[0])
            if global_heat <= 0:
                raise ValueError(f"Non-positive global heating in {path}")
            ring_heat = sp.get_tally(name="FE101 B1 domain energy deposition")
            ring_fission = sp.get_tally(name="FE101 B1 domain total fission")
            heat_ids = list(ring_heat.filters[0].bins)
            fission_ids = list(ring_fission.filters[0].bins)
            source_per_s = 250000.0 / (global_heat * openmc.data.JOULE_PER_EV)
            for i, ring in enumerate(ring_metadata):
                mid = ring["material_id"]
                heat_by_domain[k, i] = ring_heat.mean[heat_ids.index(mid), 0, 0]
                power_by_ring[k, i] = 250000.0 * ring_heat.mean[heat_ids.index(mid), 0, 0] / global_heat
                tally_fission_by_ring[k, i] = ring_fission.mean[fission_ids.index(mid), 0, 0] * source_per_s

    rows, nuclide_rate_rows, zest_rows = [], [], []
    for i, ring in enumerate(ring_metadata):
        mid = str(ring["material_id"])
        total_rate = np.zeros(len(times))
        for nuc in fission_nuclides:
            if nuc not in results[0].rates.index_nuc:
                if nuc in results[0].index_nuc:
                    _, atoms = results.get_atoms(mid, nuc)
                    if np.any(atoms > 0):
                        raise ValueError(f"Fissioning nuclide {nuc} has inventory but no saved rate")
                continue
            t, rates = results.get_reaction_rate(mid, nuc, "fission")
            assert np.allclose(t, times)
            total_rate += rates
            for k, rate in enumerate(rates):
                nuclide_rate_rows.append({"time_s": times[k], "radial_index": ring["radial_index"], "axial_index": ring["axial_index"],
                                          "nuclide": nuc, "fission_rate_per_s": float(rate)})
        inventory = {}
        for nuc in ("U235", "U238", "Pu239", "Cs137"):
            t, atoms = results.get_atoms(mid, nuc, nuc_units="atoms")
            assert np.allclose(t, times)
            inventory[nuc] = atoms
        for nuc in ("U235", "U238"):
            assert np.isclose(inventory[nuc][0], ring["initial_atoms_by_nuclide"][nuc], rtol=1e-10)
        # First-order left-endpoint integration, matching Predictor's BOS
        # transport sampling. Not an exact Bateman-integrated fission count.
        # Final_step=True supplies the genuine final rate but no extra interval.
        cumulative = np.r_[0.0, np.cumsum(total_rate[:-1] * np.diff(times))]
        energy_j = np.r_[0.0, np.cumsum(power_by_ring[:-1, i] * np.diff(times))]
        fima = cumulative / ring["initial_U_Zr_atoms"]
        burnup = energy_j / (8.64e10 * ring["initial_U_mass_kg"])
        for k, t in enumerate(times):
            row = {
                "time_s": t, "time_h": t/3600, "radial_index": ring["radial_index"], "axial_index": ring["axial_index"], "material_id": int(mid),
                "r_in_cm": ring["r_in_cm"], "r_out_cm": ring["r_out_cm"],
                "r_in_over_R": ring["r_in_over_R"], "r_out_over_R": ring["r_out_over_R"],
                "z_low_cm": ring["z_low_cm"], "z_high_cm": ring["z_high_cm"],
                "volume_cm3": ring["volume_cm3"],
                "heating_local_eV_per_source": float(heat_by_domain[k,i]),
                "initial_U_Zr_atoms": ring["initial_U_Zr_atoms"],
                "initial_U_mass_kg": ring["initial_U_mass_kg"],
                **{f"{n}_atoms": float(a[k]) for n, a in inventory.items()},
                "fission_rate_per_s": float(total_rate[k]),
                "tally_fission_rate_per_s": float(tally_fission_by_ring[k, i]),
                "cumulative_fissions": float(cumulative[k]),
                "FIMA": float(fima[k]), "FIMA_percent": float(100*fima[k]),
                "deposited_power_W": float(power_by_ring[k, i]),
                "cumulative_deposited_energy_J": float(energy_j[k]),
                "BU_MWd_kgU": float(burnup[k]),
            }
            rows.append(row)
            zest_rows.append({key: row[key] for key in ("time_s", "time_h", "radial_index", "axial_index", "z_low_cm", "z_high_cm", "r_in_cm", "r_out_cm",
                             "r_in_over_R", "r_out_over_R", "FIMA")})
    output = run_directory / "postprocessing"
    output.mkdir(exist_ok=True)
    pd.DataFrame(rows).sort_values(["time_s", "radial_index", "axial_index"]).to_csv(output/"domain_history.csv", index=False)
    pd.DataFrame(nuclide_rate_rows).to_csv(output/"fission_rates_by_nuclide.csv", index=False)
    pd.DataFrame(zest_rows).sort_values(["time_s", "radial_index", "axial_index"]).to_csv(output/"zest_FIMA_2D.csv", index=False)
    payload = {
        "geometry_baseline": "physical B", "primary_ZEST_variable": "FIMA (fraction)",
        "fima_definition": "cumulative fissions / fixed initial U+Zr atoms; H excluded",
        "integration": "first-order BOS/left-endpoint; time-discretization approximation",
        "burnup_definition": "energy deposited in ring / initial uranium mass; MWd/kgU",
        "energy_normalization": "250 kW whole-reactor heating-local; no fixed energy/fission",
        "uncertainties": "Monte Carlo/depletion/time-integration errors not propagated into cumulative outputs",
        "domains": ring_metadata, "history": sorted(rows, key=lambda r: (r["time_s"], r["radial_index"], r["axial_index"])),
    }
    (output/"domain_history.json").write_text(json.dumps(payload, indent=2)+"\n")
    assert len(rows)==525
    transfer = pd.read_csv(output/"zest_FIMA_2D.csv")
    assert len(transfer)==525 and not transfer.isna().any().any()
    assert transfer.groupby("time_h").size().eq(25).all()
    assert np.allclose(transfer.FIMA, pd.DataFrame(rows).sort_values(["time_s", "radial_index", "axial_index"]).FIMA)
    print(f"Saved {len(rows)} domain/time rows to {output}")
    return pd.DataFrame(rows)


In [ ]:
"""Future post-run comparisons; never invoked during preparation."""
def aggregate_history(history, keys):
    import numpy as np
    columns=['volume_cm3','initial_U_Zr_atoms','initial_U_mass_kg','cumulative_fissions',
             'cumulative_deposited_energy_J','fission_rate_per_s','deposited_power_W',
             'U235_atoms','U238_atoms','Pu239_atoms','Cs137_atoms']
    grouped=history.groupby(keys,as_index=False)[columns].sum()
    grouped['FIMA']=grouped.cumulative_fissions/grouped.initial_U_Zr_atoms
    grouped['FIMA_percent']=100*grouped.FIMA
    grouped['BU_MWd_kgU']=grouped.cumulative_deposited_energy_J/(8.64e10*grouped.initial_U_mass_kg)
    weighted=history.assign(weighted_FIMA=history.FIMA*history.initial_U_Zr_atoms,
                            weighted_BU=history.BU_MWd_kgU*history.initial_U_mass_kg)
    independent=weighted.groupby(keys,as_index=False)[['weighted_FIMA','weighted_BU']].sum()
    assert np.allclose(grouped.FIMA,independent.weighted_FIMA/grouped.initial_U_Zr_atoms,rtol=1e-12,atol=1e-18)
    assert np.allclose(grouped.BU_MWd_kgU,independent.weighted_BU/grouped.initial_U_mass_kg,rtol=1e-12,atol=1e-15)
    return grouped


def compare_benchmarks(run_directory, preparation_audit):
    import json
    from pathlib import Path
    import numpy as np
    import pandas as pd
    run=Path(run_directory)
    meta=json.loads(Path(preparation_audit).read_text())
    out=run/'postprocessing'
    history=pd.read_csv(out/'domain_history.csv')
    assert len(history)==525
    rings=aggregate_history(history,['time_h','radial_index'])
    whole=aggregate_history(history,['time_h'])
    axial=aggregate_history(history,['time_h','axial_index'])
    rings.to_csv(out/'radial_aggregated_history.csv',index=False)
    axial.to_csv(out/'axial_aggregated_history.csv',index=False)
    whole.to_csv(out/'FE_mean_history.csv',index=False)
    reference_path=Path(preparation_audit).parent/meta['radial_benchmark']
    reference=pd.DataFrame(json.loads(reference_path.read_text())['history']).rename(columns={'ring':'radial_index'})
    compare=rings.merge(reference,on=['time_h','radial_index'],suffixes=('_2D','_radial'),validate='one_to_one')
    assert len(compare)==105
    for field in ['initial_U_Zr_atoms','initial_U_mass_kg','volume_cm3']:
        assert np.allclose(compare[field+'_2D'],compare[field+'_radial'],rtol=1e-12)
    for field in ['FIMA','BU_MWd_kgU','fission_rate_per_s','deposited_power_W','U235_atoms','U238_atoms','Pu239_atoms','Cs137_atoms']:
        denominator=compare[field+'_radial']
        compare[field+'_relative_difference']=np.divide(compare[field+'_2D']-denominator,denominator,
            out=np.full(len(compare),np.nan),where=denominator.to_numpy()!=0)
    compare.to_csv(out/'benchmark_radial.csv',index=False)
    static_path=Path(preparation_audit).parent/meta['static_benchmark']
    static=json.loads(static_path.read_text())
    static_f=np.asarray(static['matrix_normalized_to_FE_mean'])
    static_power=np.asarray(static['matrix_power_normalized_to_FE_mean'])
    assert static_f.shape==static_power.shape==(5,5)
    initial=history[history.time_h==0].sort_values(['radial_index','axial_index']).copy()
    initial['fission_density']=initial.fission_rate_per_s/initial.volume_cm3
    initial['power_density']=initial.deposited_power_W/initial.volume_cm3
    # FIMA is zero initially; compare its derivative rate/N0, not 0/0.
    initial['initial_FIMA_slope_per_s']=initial.fission_rate_per_s/initial.initial_U_Zr_atoms
    for field in ['fission_density','power_density','initial_FIMA_slope_per_s']:
        initial[field+'_normalized']=initial[field]/np.average(initial[field],weights=initial.volume_cm3)
    initial['static_fission_normalized']=static_f.ravel()
    initial['static_power_normalized']=static_power.ravel()
    initial['fission_pattern_relative_difference']=initial.fission_density_normalized/initial.static_fission_normalized-1
    initial['power_pattern_relative_difference']=initial.power_density_normalized/initial.static_power_normalized-1
    initial['FIMA_slope_pattern_relative_difference']=initial.initial_FIMA_slope_per_s_normalized/initial.static_fission_normalized-1
    initial.to_csv(out/'benchmark_static_initial.csv',index=False)
    summary={'weighted_FE_reconstruction_passed':True,'radial_comparison_rows':len(compare),
             'static_comparison_rows':len(initial),'initial_FIMA_comparison':'Use dFIMA/dt=initial fission rate / fixed initial U+Zr atoms; FIMA(0)=0',
             'interpretation':'Report differences, do not require identity: 2D depletion and Monte Carlo sampling can differ from radial-only results.',
             'uncertainties':'No propagated inter-bin covariance or cumulative uncertainty; differences are diagnostic, not a formal significance test.'}
    (out/'benchmark_summary.json').write_text(json.dumps(summary,indent=2)+'\n')
    return summary


In [ ]:
if RUN_POSTPROCESS:
    domain_history=postprocess_2d(RUN_DIR,PREPARED_DIR/'preparation_audit.json')
else:
    print('Postprocessing disabled until actual 5x5 results exist.')
if RUN_BENCHMARKS:
    benchmark_summary=compare_benchmarks(RUN_DIR,PREPARED_DIR/'preparation_audit.json')
else:
    print('Post-run benchmarks disabled.')
